# DC Motor Predictive Maintenance — Model Training

This notebook trains an AI model to predict your motor's fault level (Normal → Watch → Warning → Critical → Failure) from MPU6050 vibration and DS18B20 temperature readings.

**How to use this in Google Colab:**
1. Go to [colab.research.google.com](https://colab.research.google.com) → File → Upload notebook → upload this `.ipynb` file
2. Run each cell top to bottom (Shift+Enter)
3. When asked, upload `dc_motor_predictive_maintenance_dataset.csv`


In [ ]:
# Step 1: Install/import what we need
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries loaded successfully.")

## Step 2: Upload your dataset
Run the cell below, then click **Choose Files** and select `dc_motor_predictive_maintenance_dataset.csv`.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)

print(f"Loaded {len(df)} rows.")
df.head()

## Step 3: Quick look at the data
Check how many examples we have of each fault level.

In [ ]:
print(df['fault_name'].value_counts())

plt.figure(figsize=(7,4))
order = ['NORMAL', 'WATCH', 'WARNING', 'CRITICAL', 'FAILURE']
sns.countplot(data=df, x='fault_name', order=order, hue='fault_name', palette='viridis', legend=False)
plt.title('Number of samples per fault level')
plt.xlabel('Fault Level')
plt.ylabel('Count')
plt.show()

## Step 4: Choose features (inputs) and target (what we're predicting)

**Features** = the sensor readings the model gets to see (just like your ESP32 does in real time).
**Target** = `fault_name`, the label we want the model to predict.


In [ ]:
feature_columns = [
    'accel_x_g', 'accel_y_g', 'accel_z_g',
    'gyro_x_dps', 'gyro_y_dps', 'gyro_z_dps',
    'vibration_rms_g', 'temperature_c'
]

X = df[feature_columns]
y = df['fault_name']

print("Features shape:", X.shape)
print("Target classes:", y.unique())

## Step 5: Split into training and testing sets
The model learns from 80% of the data, and we test it on the remaining 20% it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

## Step 6: Train the model
We're using a **Random Forest** — a reliable, beginner-friendly model that works well on sensor data and is easy to explain in a project report.

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    class_weight='balanced'  # handles the fact that Normal/Watch have more samples than Failure
)

model.fit(X_train, y_train)
print("Model trained successfully!")

## Step 7: Evaluate the model
How accurate is it, and where does it get confused?

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy*100:.2f}%\n")

print("Detailed performance per class:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix: rows = actual fault level, columns = predicted fault level
order = ['NORMAL', 'WATCH', 'WARNING', 'CRITICAL', 'FAILURE']
cm = confusion_matrix(y_test, y_pred, labels=order)

plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=order, yticklabels=order)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — where the model gets it right vs. confused')
plt.show()

## Step 8: Which sensor readings matter most?
This tells you which features the model relies on most to make decisions — useful to mention in your report.

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_columns).sort_values(ascending=False)

plt.figure(figsize=(7,4))
importances.plot(kind='barh', color='teal')
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()

print(importances)

## Step 9: Save your trained model
This downloads a `.pkl` file you can keep, reuse, or later load into another script.

In [ ]:
import joblib

joblib.dump(model, 'dc_motor_fault_model.pkl')
files.download('dc_motor_fault_model.pkl')

print("Model saved and downloaded as dc_motor_fault_model.pkl")

## What's next?

- This model currently predicts fault level from a *single* sensor reading. For true **predictive** maintenance (catching failure *before* it happens), the next upgrade is to feed it a short *sequence* of recent readings (e.g. the last 10) instead of just one — so it can learn rising trends, not just snapshots.
- If you ever want this model to run directly on your ESP32 (instead of in the cloud/Colab), it can be converted to TensorFlow Lite Micro — a bigger step, but doable as a stretch goal.
- For your report, the confusion matrix and feature importance chart above are good figures to include.
